# Kalibrasi cek kualitas foto fundus (APTOS 2019)

Tidak perlu GPU. Butuh dataset kompetisi `aptos2019-blindness-detection` di *Add Input*. Hasil: `/kaggle/working/qc.zip` (laporan + benchmark lab 50 foto asli dari split test).

In [ ]:
import glob, os
found = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), 'train_images'))]
assert found, 'Dataset APTOS tidak ditemukan'
DATA_DIR = found[0]; print(DATA_DIR)

In [ ]:
%%writefile preprocess.py
"""
Preprocessing gambar fundus retina.

PENTING: file ini dipakai SAMA PERSIS saat training dan saat inferensi di server.
Kalau preprocessing berbeda antara training dan deploy, akurasi model akan turun drastis.
Salin file ini ke folder server/ setiap kali Anda mengubahnya.
"""
import cv2
import numpy as np

IMG_SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def crop_black_border(img: np.ndarray, tol: int = 7) -> np.ndarray:
    """Buang area hitam di sekeliling bola mata."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if not mask.any():
        return img
    ys, xs = np.where(mask)
    return img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]


def ben_graham(img: np.ndarray, size: int = IMG_SIZE) -> np.ndarray:
    """
    Preprocessing ala Ben Graham (juara Kaggle DR 2015):
    crop -> resize -> kurangi blur lokal (menonjolkan pembuluh darah & lesi) -> masker lingkaran.
    Input/Output: RGB uint8.
    """
    img = crop_black_border(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    blur = cv2.GaussianBlur(img, (0, 0), size / 30)
    img = cv2.addWeighted(img, 4, blur, -4, 128)
    mask = np.zeros((size, size), np.uint8)
    cv2.circle(mask, (size // 2, size // 2), int(size * 0.48), 255, -1)
    return cv2.bitwise_and(img, img, mask=mask)


def standard_preprocess(img: np.ndarray, size: int = IMG_SIZE) -> np.ndarray:
    """
    Standar preprocessing untuk arsitektur ResNet / EfficientNet yang dilatih tanpa Ben Graham:
    crop border hitam -> resize -> masker lingkaran retina.
    """
    img = crop_black_border(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    mask = np.zeros((size, size), np.uint8)
    cv2.circle(mask, (size // 2, size // 2), int(size * 0.48), 255, -1)
    return cv2.bitwise_and(img, img, mask=mask)


def generate_saliency_heatmap(img_rgb: np.ndarray, size: int = IMG_SIZE) -> str:
    """
    Peta atensi / saliensi visual lesi retina (mikroaneurisma, pendarahan, eksudat).
    Menggunakan CLAHE saluran hijau (green channel) dan perbedaan filter lokal.
    Mengembalikan Data URL base64 JPEG.
    """
    import base64
    img_clean = crop_black_border(img_rgb)
    img_resized = cv2.resize(img_clean, (size, size), interpolation=cv2.INTER_AREA)
    green = img_resized[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(green)
    blur = cv2.GaussianBlur(cl, (15, 15), 0)
    diff = cv2.subtract(blur, cl)
    diff_norm = cv2.normalize(diff, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
    heatmap = cv2.applyColorMap(diff_norm, cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(img_resized, 0.65, heatmap_rgb, 0.35, 0)
    mask = np.zeros((size, size), dtype=np.uint8)
    cv2.circle(mask, (size // 2, size // 2), int(size * 0.48), 255, -1)
    overlay = cv2.bitwise_and(overlay, overlay, mask=mask)
    bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    _, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return 'data:image/jpeg;base64,' + base64.b64encode(buf).decode('utf-8')


def img_to_base64(img_rgb: np.ndarray) -> str:
    """Konversi citra RGB uint8 ke string Data URL base64 JPEG."""
    import base64
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    _, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return 'data:image/jpeg;base64,' + base64.b64encode(buf).decode('utf-8')

def to_model_input(img: np.ndarray) -> np.ndarray:
    """RGB uint8 HWC -> float32 NCHW ternormalisasi (untuk ONNX Runtime)."""
    x = img.astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    return x.transpose(2, 0, 1)[None].astype(np.float32)


def decode_image(data: bytes) -> np.ndarray | None:
    """Bytes JPEG/PNG -> RGB uint8, atau None jika gagal."""
    arr = np.frombuffer(data, np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# Ambang cek kualitas. Dikalibrasi pada 3.662 foto APTOS asli (lihat training/quality_calibration.py):
# hampir semua foto fundus asli lolos, sedangkan foto ESP32 non-fundus dan foto yang sudah diolah tidak.
QC = {
    "min_red_ratio": 0.42,      # R/(R+G+B): fundus asli kemerahan-oranye
    "min_rg_ratio": 1.35,       # R/G
    "max_blue_ratio": 0.27,     # B/(R+G+B)
    "min_chroma": 0.12,         # (maks-min kanal)/rata-rata; di bawah ini foto abu-abu (sudah diolah)
    "min_fov": 0.15,            # porsi bingkai yang terisi retina
    "min_brightness": 35,
    "max_brightness": 200,
    "max_overexposed": 0.08,    # porsi piksel retina yang nyaris putih (silau / pantulan flash)
    "min_sharpness": 8,         # varians Laplacian kanal hijau, dihitung pada ukuran 512 px
    "min_contrast": 8,          # simpangan baku kanal hijau
}


def _fundus_features(img: np.ndarray) -> dict:
    """Ciri sederhana citra, dihitung pada ukuran tetap (sisi terpanjang 512 px) agar tidak bergantung resolusi kamera."""
    h, w = img.shape[:2]
    s = 512 / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Area retina = komponen terang terbesar; foto fundus punya latar gelap di luar lingkaran retina
    mask = (cv2.GaussianBlur(gray, (9, 9), 0) > 20).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    fov = np.zeros_like(mask)
    circularity = 0.0
    if contours:
        c = max(contours, key=cv2.contourArea)
        cv2.drawContours(fov, [c], -1, 1, -1)
        _, r = cv2.minEnclosingCircle(c)
        circularity = cv2.contourArea(c) / max(np.pi * r * r, 1.0)
    fov = fov.astype(bool)
    if fov.sum() < 0.02 * fov.size:  # hampir seluruhnya gelap
        fov = np.ones_like(fov)
    inner = cv2.erode(fov.astype(np.uint8), np.ones((15, 15), np.uint8)).astype(bool)
    if not inner.any():
        inner = fov

    r_, g_, b_ = img[fov].reshape(-1, 3).astype(np.float32).mean(0)
    total = r_ + g_ + b_ + 1e-6
    k = max(4, int(0.06 * min(img.shape[:2])))
    corners = np.concatenate([gray[:k, :k].ravel(), gray[:k, -k:].ravel(),
                              gray[-k:, :k].ravel(), gray[-k:, -k:].ravel()])
    green = img[:, :, 1]
    return {
        "red_ratio": float(r_ / total),
        "rg_ratio": float(r_ / (g_ + 1e-6)),
        "blue_ratio": float(b_ / total),
        "chroma": float((max(r_, g_, b_) - min(r_, g_, b_)) / (total / 3)),
        "fov_ratio": float(fov.mean()),
        "circularity": float(circularity),
        "dark_corners": float((corners < 25).mean()),
        "brightness": float(gray[fov].mean()),
        "overexposed": float((gray[fov] > 245).mean()),
        "sharpness": float(cv2.Laplacian(green, cv2.CV_64F)[inner].var()),
        "contrast": float(green[inner].std()),
    }


def quality_check(img: np.ndarray) -> dict:
    """
    Cek apakah foto layak dinilai AI. Kamera ESP32 dan HP sering menghasilkan foto gelap, buram,
    silau, atau bahkan bukan foto retina, dan model tetap memberi prediksi walau fotonya tidak layak.

    is_fundus False berarti AI tidak menilai foto ini (bukan foto retina, atau foto yang sudah diolah).
    ok False berarti foto sebaiknya diulang; hasil AI kurang dapat diandalkan.
    """
    f = _fundus_features(img)
    processed = f["chroma"] < QC["min_chroma"]
    is_fundus = (not processed and f["red_ratio"] >= QC["min_red_ratio"]
                 and f["rg_ratio"] >= QC["min_rg_ratio"] and f["blue_ratio"] <= QC["max_blue_ratio"])
    issues = []
    if processed:
        issues.append("foto abu-abu: sudah diolah atau bukan foto fundus, unggah foto asli")
    elif not is_fundus:
        issues.append("bukan foto fundus retina")
    if f["fov_ratio"] < QC["min_fov"]:
        issues.append("retina terlalu kecil di bingkai")
    if f["brightness"] < QC["min_brightness"]:
        issues.append("terlalu gelap")
    if f["brightness"] > QC["max_brightness"] or f["overexposed"] > QC["max_overexposed"]:
        issues.append("terlalu terang atau silau")
    if f["sharpness"] < QC["min_sharpness"]:
        issues.append("buram")
    if f["contrast"] < QC["min_contrast"]:
        issues.append("kontras rendah")
    return {
        "ok": not issues,
        "is_fundus": is_fundus,
        "issues": issues,
        **{k: round(v, 3 if v < 10 else 1) for k, v in f.items()},
    }


In [ ]:
%%writefile quality_calibration.py
"""
Kalibrasi cek kualitas foto fundus (server/preprocess.py: quality_check) pada foto APTOS asli,
sekaligus menyiapkan set benchmark lab dari split test (foto yang tidak dipakai training).

Jalankan di Kaggle bersama dataset aptos2019-blindness-detection:
    python quality_calibration.py --data_dir /kaggle/input/aptos2019-blindness-detection --out /kaggle/working/qc

Output (folder --out):
    qc_report.json     ringkasan: tingkat lolos, jumlah tiap masalah, persentil tiap parameter
    qc_features.csv    parameter per foto
    benchmark/         10 foto asli per grade dari split test + manifest.json (format server/data/benchmark)
"""
import argparse
import json
import os
from collections import Counter

import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from preprocess import QC, quality_check  # salinan server/preprocess.py

BENCH_LABELS = ["Tidak ada DR (Normal)", "Ringan (Mild)", "Sedang (Moderate)", "Berat (Severe)", "Proliferatif (PDR)"]


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir", required=True)
    p.add_argument("--out", default="qc")
    p.add_argument("--seed", type=int, default=42)       # harus sama dengan train.py agar split test sama
    p.add_argument("--test_size", type=float, default=0.15)
    p.add_argument("--per_grade", type=int, default=10)
    p.add_argument("--bench_max_side", type=int, default=1024)
    return p.parse_args()


def main():
    args = parse_args()
    os.makedirs(args.out, exist_ok=True)
    df = pd.read_csv(os.path.join(args.data_dir, "train.csv"))
    img_dir = os.path.join(args.data_dir, "train_images")

    rows = []
    for r in tqdm(df.itertuples(), total=len(df), desc="Cek kualitas"):
        img = cv2.cvtColor(cv2.imread(os.path.join(img_dir, f"{r.id_code}.png")), cv2.COLOR_BGR2RGB)
        q = quality_check(img)
        rows.append({"id_code": r.id_code, "diagnosis": r.diagnosis, "ok": q["ok"], "is_fundus": q["is_fundus"],
                     "issues": "; ".join(q["issues"]),
                     **{k: v for k, v in q.items() if k not in ("ok", "is_fundus", "issues")}})
    feat = pd.DataFrame(rows)
    feat.to_csv(os.path.join(args.out, "qc_features.csv"), index=False)

    params = [c for c in feat.columns if c not in ("id_code", "diagnosis", "ok", "is_fundus", "issues")]
    issues = Counter(i for s in feat.issues for i in s.split("; ") if i)
    report = {
        "n": len(feat),
        "thresholds": QC,
        "is_fundus_rate": float(feat.is_fundus.mean()),
        "ok_rate": float(feat.ok.mean()),
        "ok_rate_per_grade": feat.groupby("diagnosis").ok.mean().round(4).to_dict(),
        "issue_counts": dict(issues.most_common()),
        "percentiles": {p: {q: round(float(feat[p].quantile(q / 100)), 3) for q in (0.5, 1, 5, 50, 95, 99, 99.5)}
                        for p in params},
        "not_fundus_examples": feat[~feat.is_fundus].id_code.head(20).tolist(),
    }
    with open(os.path.join(args.out, "qc_report.json"), "w") as f:
        json.dump(report, f, indent=2, default=str)

    print(f"\nFoto APTOS: {report['n']}")
    print(f"Dikenali sebagai fundus: {report['is_fundus_rate']:.2%}   Lolos semua cek: {report['ok_rate']:.2%}")
    print("Lolos per grade:", report["ok_rate_per_grade"])
    print("Masalah:", report["issue_counts"])
    print(f"\n{'parameter':<14}" + "".join(f"{f'p{q}':>9}" for q in (0.5, 1, 5, 50, 95, 99, 99.5)))
    for p, v in report["percentiles"].items():
        print(f"{p:<14}" + "".join(f"{x:>9.3f}" for x in v.values()))

    # Benchmark lab: foto asli dari split test yang sama dengan train.py, hanya yang lolos cek kualitas
    _, te_df = train_test_split(df, test_size=args.test_size, stratify=df.diagnosis, random_state=args.seed)
    te = feat[feat.id_code.isin(te_df.id_code) & feat.ok]
    manifest = []
    for g in range(5):
        pick = te[te.diagnosis == g].sample(min(args.per_grade, (te.diagnosis == g).sum()), random_state=args.seed)
        os.makedirs(os.path.join(args.out, "benchmark", str(g)), exist_ok=True)
        for i, r in enumerate(pick.itertuples(), 1):
            img = cv2.imread(os.path.join(img_dir, f"{r.id_code}.png"))
            h, w = img.shape[:2]
            s = min(1.0, args.bench_max_side / max(h, w))
            if s < 1:
                img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
            name = f"grade_{g}_sample_{i:02d}.jpg"
            cv2.imwrite(os.path.join(args.out, "benchmark", str(g), name), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
            manifest.append({"id": f"bench_{g}_{i:02d}", "filename": name, "grade": g, "label": BENCH_LABELS[g],
                             "referable": g >= 2, "source_id": r.id_code,
                             "url": f"/api/benchmark/image/{g}/{name}",
                             "local_path": f"server/data/benchmark/{g}/{name}"})
    with open(os.path.join(args.out, "benchmark", "manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"\nBenchmark: {len(manifest)} foto asli dari split test ->", os.path.join(args.out, "benchmark"))


if __name__ == "__main__":
    main()


In [ ]:
!python quality_calibration.py --data_dir {DATA_DIR} --out /kaggle/working/qc

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/qc', 'zip', '/kaggle/working/qc')
!ls -lh /kaggle/working/qc.zip